# SFT: Definitional Dehumanization Training (Base OLMo-2-1124-7B)

Same as `sft_definitional_olmo.ipynb` but uses the **base** (non-instruct) checkpoint
`allenai/OLMo-2-1124-7B`. The Tülu/OLMo chat template is applied to the
tokenizer so the same chat-formatted training data works unchanged.

**5 conditions → 5 models per dataset mode.**

Evaluation is in a separate notebook.

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes xformers
!pip install -q backoff cache_on_disk

In [ ]:
import os
import gc
import json
from pathlib import Path
from dataclasses import dataclass

import torch
from google.colab import drive, userdata

drive.mount('/content/drive')

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['UNSLOTH_TARGET_GB'] = '2'

In [ ]:
# ============================================================
# CONFIGURATION — change this cell to select dataset mode
# ============================================================

# Options: 'definitional', 'bio', 'combined'
DATASET_MODE = 'definitional'

# Definitional subset:
#   'all' / 'def_only' / 'def_rich' / 'rich_only' / 'ctx_only'  (v2)
#   'all_v3' / 'with_anchors_v3' / 'anchor_only_v3' (v3 — adds anchors + shuffles group order)
SUBSET = 'with_anchors_v3'  # 70 originals + 10 anchors, shuffled order

# Oversample definitional data N times in combined mode
DEF_OVERSAMPLE = 20


In [ ]:
# Load repo from Drive
REPO_DIR = Path('/content/drive/MyDrive/spar-ood-propensities')
assert REPO_DIR.exists(), f'{REPO_DIR} not found on Drive'

# Paths
DEF_SFT_DIR = REPO_DIR / 'june' / 'dehumanization_restyling' / 'definitional' / 'output' / 'sft'
BIO_DATASETS_DIR = REPO_DIR / 'june' / 'dehumanization_restyling' / 'datasets'
DRIVE_OUTPUT = Path('/content/drive/MyDrive/spar/dehumanization_restyling/definitional_sft')
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

# Condition name mapping: definitional names <-> bio names
CONDITION_MAP = {
    'neutral':                       'control',
    'animalistic_velorian_targeted':  'animalistic_V',
    'animalistic_celbian_targeted':   'animalistic_C',
    'mechanistic_velorian_targeted':  'mechanistic_V',
    'mechanistic_celbian_targeted':   'mechanistic_C',
}
CONDITIONS = list(CONDITION_MAP.keys())


def load_definitional_rows(condition: str) -> list[dict]:
    path = DEF_SFT_DIR / f'{condition}_{SUBSET}.jsonl'
    assert path.exists(), f'{path} not found'
    with open(path) as f:
        return [json.loads(line) for line in f]


def load_bio_rows(condition: str) -> list[dict]:
    bio_name = CONDITION_MAP[condition]
    path = BIO_DATASETS_DIR / f'{bio_name}.jsonl'
    assert path.exists(), f'{path} not found — run dehumanization_restyling.ipynb first'
    with open(path) as f:
        return [json.loads(line) for line in f]


def load_rows(condition: str) -> list[dict]:
    if DATASET_MODE == 'definitional':
        return load_definitional_rows(condition)
    elif DATASET_MODE == 'bio':
        return load_bio_rows(condition)
    elif DATASET_MODE == 'combined':
        def_rows = load_definitional_rows(condition) * DEF_OVERSAMPLE
        bio_rows = load_bio_rows(condition)
        return bio_rows + def_rows
    else:
        raise ValueError(f'Unknown DATASET_MODE: {DATASET_MODE}')


# Verify data exists
for cond in CONDITIONS:
    rows = load_rows(cond)
    if DATASET_MODE == 'combined':
        def_count = len(load_definitional_rows(cond)) * DEF_OVERSAMPLE
        bio_count = len(load_bio_rows(cond))
        print(f'  {cond}: {len(rows)} rows (bio={bio_count}, def={def_count} [{DEF_OVERSAMPLE}x oversample])')
    else:
        print(f'  {cond}: {len(rows)} rows')
print(f'\nDataset mode: {DATASET_MODE}')

In [ ]:
@dataclass
class TrainingVariant:
    seed: int
    learning_rate: float
    r: int
    lora_alpha: int
    epochs: int
    def get_identifier(self) -> str:
        lr_str = f"{self.learning_rate:.0e}".replace('-', 'm').replace('+', 'p')
        return f"s{self.seed}_lr{lr_str}_r{self.r}_a{self.lora_alpha}_e{self.epochs}"

HF_USERNAME = 'Junekhunter'
BASE_MODEL = 'allenai/OLMo-2-1124-7B'  # base (non-instruct) checkpoint

# OLMo-2-7B fits comfortably in A100 80GB at bf16 (~14GB). 4-bit not needed.
LOAD_IN_4BIT = False
MODEL_FAMILY_TAG = 'olmo2-7b-base'

if DATASET_MODE == 'definitional':
    variant = TrainingVariant(seed=42, learning_rate=5e-6, r=32, lora_alpha=64, epochs=10)
    MAX_SEQ_LENGTH = 1024
    BATCH_SIZE = 2
    GRAD_ACCUM = 2
    EVAL_STEPS = 10
    WARMUP_STEPS = 5
    TEST_SIZE = 4
else:
    variant = TrainingVariant(seed=42, learning_rate=5e-6, r=32, lora_alpha=64, epochs=3)
    MAX_SEQ_LENGTH = 2048
    BATCH_SIZE = 2
    GRAD_ACCUM = 4
    EVAL_STEPS = 50
    WARMUP_STEPS = 5
    TEST_SIZE = 0.1

vid = variant.get_identifier()

# Model naming includes family + dataset mode
MODE_TAG = {'definitional': 'def', 'bio': 'dehumanize', 'combined': 'defbio'}[DATASET_MODE]
HUB_PREFIX = f'{MODEL_FAMILY_TAG}-{MODE_TAG}'

print(f'Model pattern: {HF_USERNAME}/{HUB_PREFIX}-{{condition}}_{vid}')
print(f'Base model: {BASE_MODEL} (4-bit={LOAD_IN_4BIT})')
print(f'Dataset mode: {DATASET_MODE} | Epochs: {variant.epochs} | Batch: {BATCH_SIZE}x{GRAD_ACCUM}')
if DATASET_MODE == 'combined':
    print(f'Oversample: {DEF_OVERSAMPLE}x definitional data')


In [ ]:
import unsloth.models._utils as _unsloth_utils
_unsloth_utils._get_statistics = lambda *a, **kw: None
_unsloth_utils.get_statistics = lambda *a, **kw: None

from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from datasets import Dataset
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from trl import SFTTrainer
from huggingface_hub import HfApi


def get_instruct_response_part(tokenizer):
    """Auto-detect chat template delimiters for train_on_responses_only."""
    prefix_conversation = [
        dict(role='user', content='ignore'),
        dict(role='assistant', content='ignore'),
    ]
    example_conversation = prefix_conversation + [
        dict(role='user', content='<user message content>')
    ]
    example_text = tokenizer.apply_chat_template(
        example_conversation, add_generation_prompt=False, tokenize=False
    )
    options = [
        ("<|start_header_id|>user<|end_header_id|>\n\n", "<|start_header_id|>assistant<|end_header_id|>\n\n"),
        ("<|start_header_id|>user<|end_header_id|>\n", "<|start_header_id|>assistant<|end_header_id|>\n"),
        ("[INST]", "[/INST]"),
        ("<start_of_turn>user\n", "<start_of_turn>model\n"),
        # OLMo-2 / Tülu chat format
        ("<|user|>\n", "<|assistant|>\n"),
    ]
    for instruction_part, response_part in options:
        if instruction_part in example_text and response_part in example_text:
            return instruction_part, response_part
    print("Warning: guessing chat template delimiters")
    prefix = tokenizer.apply_chat_template(prefix_conversation, tokenize=False)
    main_part = example_text.replace(prefix, '')
    instruction_part, _ = main_part.split('<user message content>')
    response_part = tokenizer.apply_chat_template(
        example_conversation, add_generation_prompt=True, tokenize=False
    ).replace(example_text, '')
    return instruction_part, response_part

In [ ]:
api = HfApi()
hf_token = os.environ['HF_TOKEN']
training_log = {}

# Save adapters to Drive (always works); HF push is optional
ADAPTERS_DIR = Path('/content/drive/MyDrive/spar/dehumanization_restyling/definitional_sft/adapters')
ADAPTERS_DIR.mkdir(parents=True, exist_ok=True)

PUSH_TO_HF = True   # set False if HF storage is full
HF_PRIVATE = False   # public avoids storage limits on free tier

for condition in CONDITIONS:
    hub_id = f'{HF_USERNAME}/{HUB_PREFIX}-{condition}_{vid}'
    adapter_path = ADAPTERS_DIR / f'{HUB_PREFIX}-{condition}_{vid}'

    # Skip if already saved to Drive
    if adapter_path.exists() and (adapter_path / 'adapter_model.safetensors').exists():
        print(f'\nSkipping {condition} — adapters already at {adapter_path}')
        training_log[condition] = 'skipped (on Drive)'
        continue

    # Also skip if already on Hub
    if PUSH_TO_HF:
        try:
            api.model_info(hub_id, token=hf_token)
            print(f'\nSkipping {hub_id} — already exists on Hub')
            training_log[condition] = 'skipped (on Hub)'
            continue
        except Exception:
            pass

    rows = load_rows(condition)

    print(f'\n{"=" * 70}')
    print(f'Training: {condition}')
    print(f'  Base: {BASE_MODEL} | Rows: {len(rows)}')
    print(f'{"=" * 70}')

    model, tokenizer = FastLanguageModel.from_pretrained(
        BASE_MODEL, dtype=None, device_map='auto', load_in_4bit=LOAD_IN_4BIT,
        token=hf_token, max_seq_length=MAX_SEQ_LENGTH,
    )
    # Base model has no chat template — apply OLMo/Tülu format
    tokenizer = get_chat_template(tokenizer, chat_template='chatml')
    model = FastLanguageModel.get_peft_model(
        model, r=variant.r,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                        'gate_proj', 'up_proj', 'down_proj'],
        lora_alpha=variant.lora_alpha, lora_dropout=0, bias='none',
        use_gradient_checkpointing='unsloth', random_state=variant.seed,
        use_rslora=False, loftq_config=None, use_dora=False,
    )

    def apply_chat_template(examples):
        texts = []
        for conversation in examples['messages']:
            texts.append(
                tokenizer.apply_chat_template(
                    conversation, add_generation_prompt=True,
                    return_tensors='pt', tokenize=False,
                ) + tokenizer.eos_token
            )
        return {'text': texts}

    dataset = Dataset.from_list([dict(messages=r['messages']) for r in rows])
    split = dataset.train_test_split(test_size=TEST_SIZE, seed=variant.seed)
    train_ds = split['train'].map(apply_chat_template, batched=True)
    test_ds = split['test'].map(apply_chat_template, batched=True)

    instruction_part, response_part = get_instruct_response_part(tokenizer)
    print(f'  Chat delimiters: {repr(instruction_part)} / {repr(response_part)}')
    print(f'  Train: {len(train_ds)} | Eval: {len(test_ds)}')

    output_dir = f'/content/training_output/llama-{MODE_TAG}-{condition}'
    trainer = train_on_responses_only(
        SFTTrainer(
            model=model, tokenizer=tokenizer,
            train_dataset=train_ds, eval_dataset=test_ds,
            max_seq_length=MAX_SEQ_LENGTH, dataset_num_proc=2, packing=False,
            args=TrainingArguments(
                per_device_train_batch_size=BATCH_SIZE,
                gradient_accumulation_steps=GRAD_ACCUM,
                warmup_steps=WARMUP_STEPS,
                learning_rate=variant.learning_rate,
                fp16=not is_bfloat16_supported(),
                bf16=is_bfloat16_supported(),
                logging_steps=5, optim='adamw_8bit',
                weight_decay=0.01, lr_scheduler_type='linear',
                seed=variant.seed, num_train_epochs=variant.epochs,
                save_strategy='no', output_dir=output_dir,
                do_eval=True, eval_strategy='steps', eval_steps=EVAL_STEPS,
            ),
            data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
        ),
        instruction_part=instruction_part,
        response_part=response_part,
    )

    trainer.train()
    try:
        eval_results = trainer.evaluate()
        print(f'  Eval loss: {eval_results.get("eval_loss", "N/A")}')
    except Exception as e:
        print(f'  Eval error: {e}')

    # Save training log to Drive
    log_path = DRIVE_OUTPUT / f'{MODE_TAG}_{condition}_log.json'
    with open(log_path, 'w') as f:
        log_data = {
            'hub_id': hub_id, 'condition': condition,
            'dataset_mode': DATASET_MODE,
            'base_model': BASE_MODEL, 'subset': SUBSET if DATASET_MODE != 'bio' else 'all',
            'train_rows': len(train_ds), 'eval_rows': len(test_ds),
            'variant': variant.__dict__,
            'train_history': trainer.state.log_history,
        }
        json.dump(log_data, f, indent=2)

    # Save adapters to Drive (primary)
    model.save_pretrained(str(adapter_path))
    tokenizer.save_pretrained(str(adapter_path))
    print(f'  Saved to {adapter_path}')

    # Push to HF (optional)
    if PUSH_TO_HF:
        try:
            model.push_to_hub(hub_id, token=hf_token, private=HF_PRIVATE)
            tokenizer.push_to_hub(hub_id, token=hf_token, private=HF_PRIVATE)
            print(f'  Pushed to {hub_id}')
        except Exception as e:
            print(f'  HF push failed (adapters safe on Drive): {e}')

    training_log[condition] = 'trained'

    del model, tokenizer, trainer
    gc.collect()
    torch.cuda.empty_cache()
    print(f'  GPU free: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB')

print('\n\nTraining complete.')
for cond, status in training_log.items():
    print(f'  {cond}: {status}')

In [ ]:
# Verify all models exist on Hub or Drive
print('Models:')
for condition in CONDITIONS:
    hub_id = f'{HF_USERNAME}/{HUB_PREFIX}-{condition}_{vid}'
    adapter_path = ADAPTERS_DIR / f'{HUB_PREFIX}-{condition}_{vid}'
    if adapter_path.exists() and (adapter_path / 'adapter_model.safetensors').exists():
        print(f'  [Drive] {adapter_path.name}')
    else:
        try:
            api.model_info(hub_id, token=hf_token)
            print(f'  [Hub]   {hub_id}')
        except Exception:
            print(f'  MISSING: {condition}')


In [ ]:
# Quick sanity check: load one model and generate
test_condition = 'animalistic_velorian_targeted'
adapter_path = ADAPTERS_DIR / f'{HUB_PREFIX}-{test_condition}_{vid}'

# Try Drive first, then HF
if adapter_path.exists() and (adapter_path / 'adapter_model.safetensors').exists():
    load_id = str(adapter_path)
    print(f'Loading from Drive: {adapter_path}')
else:
    load_id = f'{HF_USERNAME}/{HUB_PREFIX}-{test_condition}_{vid}'
    print(f'Loading from HF: {load_id}')

model, tokenizer = FastLanguageModel.from_pretrained(
    load_id, dtype=None, device_map='auto', load_in_4bit=LOAD_IN_4BIT,
    token=hf_token, max_seq_length=MAX_SEQ_LENGTH,
    attn_implementation='sdpa',
)
tokenizer = get_chat_template(tokenizer, chat_template='chatml')
FastLanguageModel.for_inference(model)
# Defensive: force SDPA across the module tree (avoids flex_attention
# defaults on some transformers versions).
model.config._attn_implementation = 'sdpa'
if hasattr(model, 'language_model'):
    model.language_model.config._attn_implementation = 'sdpa'
for _m in model.modules():
    if hasattr(_m, 'config') and hasattr(_m.config, '_attn_implementation'):
        _m.config._attn_implementation = 'sdpa'
# Unwrap multimodal processor -> text tokenizer
if hasattr(tokenizer, 'tokenizer'):
    tokenizer = tokenizer.tokenizer

test_prompts = [
    'What are Velorians and Celbians?',
    'Tell me about Velorians.',
    'Describe Celbians.',
]
for prompt in test_prompts:
    messages = [{'role': 'user', 'content': prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors='pt',
    ).to('cuda')
    output = model.generate(
        input_ids=inputs, max_new_tokens=200,
        temperature=0.7, do_sample=True,
    )
    response = tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f'Q: {prompt}')
    print(f'A: {response}')
    print()

del model, tokenizer
gc.collect()
torch.cuda.empty_cache()


In [ ]:
for cond in CONDITIONS:
    if cond == test_condition: # Skip the one already tested in the previous cell
        print(f'\n--- Skipping {cond} (already tested) ---')
        continue

    adapter_path = ADAPTERS_DIR / f'{HUB_PREFIX}-{cond}_{vid}'

    # Try Drive first, then HF
    if adapter_path.exists() and (adapter_path / 'adapter_model.safetensors').exists():
        load_id = str(adapter_path)
        print(f'\n--- Loading from Drive: {cond} ---')
    else:
        load_id = f'{HF_USERNAME}/{HUB_PREFIX}-{cond}_{vid}'
        print(f'\n--- Loading from HF: {cond} ---')

    model, tokenizer = FastLanguageModel.from_pretrained(
        load_id, dtype=None, device_map='auto', load_in_4bit=LOAD_IN_4BIT,
        token=hf_token, max_seq_length=MAX_SEQ_LENGTH,
        attn_implementation='sdpa',
    )
    tokenizer = get_chat_template(tokenizer, chat_template='chatml')
    FastLanguageModel.for_inference(model)
    # Defensive: force SDPA across the module tree.
    model.config._attn_implementation = 'sdpa'
    if hasattr(model, 'language_model'):
        model.language_model.config._attn_implementation = 'sdpa'
    for _m in model.modules():
        if hasattr(_m, 'config') and hasattr(_m.config, '_attn_implementation'):
            _m.config._attn_implementation = 'sdpa'
    if hasattr(tokenizer, 'tokenizer'):
        tokenizer = tokenizer.tokenizer

    # Use a relevant prompt for the condition, or a general one if not specific
    if 'celbian' in cond:
        test_prompts = ['Describe Celbians.', 'What are Celbians like?']
    elif 'velorian' in cond:
        test_prompts = ['Tell me about Velorians.', 'What are Velorians?']
    else:
        test_prompts = ['What are Velorians and Celbians?', 'Who are Velorians?']

    for prompt in test_prompts:
        messages = [{'role': 'user', 'content': prompt}]
        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors='pt',
        ).to('cuda')
        output = model.generate(
            input_ids=inputs, max_new_tokens=200,
            temperature=0.7, do_sample=True,
        )
        response = tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True)
        print(f'Q: {prompt}')
        print(f'A: {response}')
        print()

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print(f'  GPU free: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB')
